## Bronze to Silver Vault
Builds **Data Vault 2.0** silver-layer Delta tables (Hubs, Links, Satellites) from raw CRM tables in Unity Catalog.

| Layer | Artefacts |
|---|---|
| **Hub** | hub_person, hub_natural_person, hub_legal_person, hub_product, hub_lead, hub_customer, hub_identities, hub_contact, hub_consent, hub_account, hub_marketing_preference, hub_marketing_engagement, hub_quote, hub_policy, hub_motor, hub_home, hub_home_address |
| **Link** | link_person_natural_person, link_person_legal_person, link_person_identities, link_person_contact, link_person_consent, link_person_home_address, link_person_account, link_person_lead, link_person_marketing_preference, link_person_marketing_engagement, link_customer_person, link_customer_lead, link_quote_person, link_quote_product, link_policy_customer, link_policy_product, link_product_motor, link_product_home |
| **Satellite** | sat_natural_person, sat_legal_person, sat_person, sat_lead, sat_customer, sat_identities, sat_contact, sat_consent, sat_account, sat_marketing_preference, sat_marketing_engagement, sat_quote, sat_policy, sat_motor, sat_home, sat_home_address, sat_product |

In [0]:
%run /Workspace/Allianz/Allianz_COE/ETL_Ingestion/utils/utils_nb

In [0]:
%python
dbutils.widgets.text("input_catalog", "allianz_coe")
dbutils.widgets.text("input_schema", "amit_raw_test")
dbutils.widgets.text("output_catalog", "allianz_coe")
dbutils.widgets.text("output_schema", "amit_silver_test")
dbutils.widgets.text("metadata_catalog", "allianz_coe")
dbutils.widgets.text("metadata_schema", "config")
dbutils.widgets.text("metadata_table", "brz_vault_sql_metadata")
dbutils.widgets.text("hub_load_date", "")
dbutils.widgets.text("link_load_date", "")
dbutils.widgets.text("sat_load_date", "")


In [0]:
import hashlib
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
from delta.tables import DeltaTable


def md5_hasher(value: str) -> str:
    """Return the MD5 hex digest of the given string value."""
    return hashlib.md5(str(value).encode("utf-8")).hexdigest()


def read_table_rows(catalog, schema, table_name):
    """Read all rows from a Delta table as a list of dicts (None values converted to empty string)."""
    full_name = f"`{catalog}`.`{schema}`.`{table_name}`"
    try:
        df = spark.table(full_name)
        rows = [row.asDict() for row in df.collect()]
        return [{k: (str(v) if v is not None else "") for k, v in r.items()} for r in rows]
    except Exception as e:
        print(f"  \u26a0 Table {full_name} not found or unreadable: {e}")
        return []


def write_delta_table(catalog, schema, table_name, rows, columns, key_columns=None):
    """Write rows to a Delta table using MERGE (insert-only).

    New records (by key) are inserted; existing records are skipped.
    key_columns: list of column names used to match existing records.
                 Defaults to the first column in the schema (the hash key).
    """
    quoted_name   = f"`{catalog}`.`{schema}`.`{table_name}`"
    unquoted_name = f"{catalog}.{schema}.{table_name}"
    schema_struct  = StructType([StructField(c, StringType(), True) for c in columns])

    # Build source DataFrame
    if not rows:
        source_df = spark.createDataFrame([], schema_struct)
    else:
        data = [tuple(str(row.get(c, "") or "") for c in columns) for row in rows]
        source_df = spark.createDataFrame(data, schema=schema_struct)

    if not rows:
        # Nothing to merge; ensure the table exists (empty is fine)
        if not spark.catalog.tableExists(unquoted_name):
            source_df.write.format("delta").mode("overwrite").saveAsTable(quoted_name)
        return

    if not spark.catalog.tableExists(unquoted_name):
        # First run – table does not exist yet, create it
        source_df.write.format("delta").mode("overwrite").saveAsTable(quoted_name)
    else:
        # Table exists – MERGE: insert rows whose key is not already present
        merge_keys = key_columns or [columns[0]]  # fallback to first column (hash key)
        merge_condition = " AND ".join(
            f"target.`{col}` = source.`{col}`" for col in merge_keys
        )
        (DeltaTable.forName(spark, unquoted_name)
            .alias("target")
            .merge(source_df.alias("source"), merge_condition)
            .whenNotMatchedInsertAll()
            .execute())

In [0]:
# Default catalog/schema settings
INPUT_CATALOG = "allianz_coe"
INPUT_SCHEMA = "amit_raw_test"
OUTPUT_CATALOG = "allianz_coe"
OUTPUT_SCHEMA = "amit_silver_test"
METADATA_CATALOG = "allianz_coe"
METADATA_SCHEMA = "amit_control"
METADATA_TABLE = "brz_vault_sql_metadata"


In [0]:
import json


In [0]:
def hk_map(rows, field):
    return {r[field]: md5_hasher(r[field]) for r in rows if r.get(field)}


def hub_row(pk, bid_field, bid, load_date, record_source):
    return {pk: md5_hasher(bid), "load_date": load_date, "record_source": record_source, bid_field: bid}


def link_row(pk, left_name, left_value, right_name, right_value, load_date, record_source):
    return {pk: md5_hasher(f"{left_value}|{right_value}"), "load_date": load_date, "record_source": record_source, left_name: left_value, right_name: right_value}


def dedupe(rows, fields):
    seen = set()
    out = []
    for row in rows:
        key = tuple(row.get(field, "") for field in fields)
        if key in seen:
            continue
        seen.add(key)
        out.append(row)
    return out


def ordered(rows, schema):
    return [{col: row.get(col, "") for col in schema} for row in rows]


def has_required_fields(row, fields):
    return all(row.get(field) for field in fields)


def resolve_hash_key(ids, row, ref):
    raw_value = row.get(ref["row_field"])
    if not raw_value:
        return None
    return ids[ref["hash_map"]].get(raw_value)


def metadata_full_name(catalog, schema, table):
    return f"`{catalog}`.`{schema}`.`{table}`"


def render_build_sql(build_sql, hub_load_date, link_load_date, sat_load_date, record_source):
    replacements = {
        "${hub_load_date}": f"timestamp('{hub_load_date}')" if hub_load_date else "current_timestamp()",
        "${link_load_date}": f"timestamp('{link_load_date}')" if link_load_date else "current_timestamp()",
        "${sat_load_date}": f"timestamp('{sat_load_date}')" if sat_load_date else "current_timestamp()",
        "${record_source}": f"'{record_source}'",
    }
    rendered = build_sql
    for token, value in replacements.items():
        rendered = rendered.replace(token, value)
    return rendered


def collect_rows(df):
    rows = []
    for row in df.collect():
        converted = {}
        for key, value in row.asDict().items():
            converted[key] = str(value) if value is not None else ""
        rows.append(converted)
    return rows


def normalize_source_name(value):
    return (value or "").strip().upper()


def load_source_configs(catalog, schema, table):
    plain_name = f"{catalog}.{schema}.{table}"
    if not spark.catalog.tableExists(plain_name):
        raise ValueError(f"Required metadata table {plain_name} does not exist")

    metadata_df = spark.table(metadata_full_name(catalog, schema, table)).filter("is_active = 'Y'")
    metadata_columns = set(metadata_df.columns)
    required_columns = {"record_source", "input_catalog", "input_schema", "output_catalog", "output_schema"}
    missing_columns = sorted(required_columns - metadata_columns)
    if missing_columns:
        raise ValueError(
            f"Metadata table {plain_name} must contain source config columns: {', '.join(missing_columns)}"
        )

    distinct_rows = (
        metadata_df
        .select("record_source", "input_catalog", "input_schema", "output_catalog", "output_schema")
        .dropna(subset=["record_source", "input_catalog", "input_schema", "output_catalog", "output_schema"])
        .dropDuplicates()
        .orderBy("record_source", "input_catalog", "input_schema", "output_catalog", "output_schema")
        .collect()
    )

    source_configs = []
    for row in distinct_rows:
        source_name = normalize_source_name(row["record_source"]).lower()
        source_configs.append({
            "source_name": source_name,
            "record_source": row["record_source"],
            "input_catalog": row["input_catalog"],
            "input_schema": row["input_schema"],
            "output_catalog": row["output_catalog"],
            "output_schema": row["output_schema"],
        })

    if not source_configs:
        raise ValueError(f"No active source configurations found in metadata table {plain_name}")

    return source_configs


def load_runtime_metadata(catalog, schema, table, record_source=None):
    plain_name = f"{catalog}.{schema}.{table}"
    if not spark.catalog.tableExists(plain_name):
        raise ValueError(f"Required metadata table {plain_name} does not exist")

    metadata_df = spark.table(metadata_full_name(catalog, schema, table)).filter("is_active = 'Y'")
    metadata_columns = set(metadata_df.columns)
    if record_source and "record_source" in metadata_columns:
        metadata_df = metadata_df.filter(f"upper(trim(record_source)) = '{normalize_source_name(record_source)}'")

    metadata_df = metadata_df.orderBy("process_order", "target_table")
    rows = [row.asDict() for row in metadata_df.collect()]
    if not rows:
        scope = normalize_source_name(record_source) if record_source else "ALL"
        raise ValueError(f"No active metadata rows found in {plain_name} for record_source={scope}")

    if "build_sql" in metadata_columns:
        return {
            "mode": "sql",
            "operations": rows,
        }

    raw_tables = {}
    schemas = {}
    hub_schemas = {}
    hash_key_specs = {}
    grouped = {}

    for row in rows:
        raw_tables[row["source_alias"]] = row["source_table"]
        schemas[row["target_table"]] = json.loads(row["schema_json"])

        if row["hash_alias"] and row["hash_source_field"]:
            hash_key_specs[row["hash_alias"]] = {
                "source": row["source_alias"],
                "field": row["hash_source_field"],
            }

        operation = {"type": row["object_type"], "table": row["target_table"]}
        required = json.loads(row["required_fields_json"] or "[]")
        if required:
            operation["when"] = required

        if row["object_type"] == "hub":
            hub_schemas[row["target_table"]] = (
                schemas[row["target_table"]][0],
                row["business_key_target"],
            )
            operation["key_field"] = row["business_key_source"]
        elif row["object_type"] == "sat":
            operation["hash_map"] = row["hash_alias"]
            operation["key_field"] = row["hash_source_field"]
            operation["mappings"] = json.loads(row["mappings_json"] or "{}")
        else:
            operation["pk"] = row["link_pk"]
            operation["left"] = json.loads(row["left_ref_json"])
            operation["right"] = json.loads(row["right_ref_json"])

        grouped.setdefault(row["source_alias"], []).append(operation)

    transform_specs = [
        {"source": source_alias, "operations": operations}
        for source_alias, operations in grouped.items()
    ]
    return {
        "mode": "legacy",
        "raw_tables": raw_tables,
        "schemas": schemas,
        "hub_schemas": hub_schemas,
        "hash_key_specs": hash_key_specs,
        "transform_specs": transform_specs,
    }


def apply_operation(row, operation, ids, out, hub_schemas, schemas, hub_load_date, link_load_date, sat_load_date, record_source):
    required = operation.get("when", [])
    if required and not has_required_fields(row, required):
        return

    if operation["type"] == "hub":
        key_value = row.get(operation["key_field"])
        if not key_value:
            return
        pk, bid_field = hub_schemas[operation["table"]]
        out[operation["table"]].append(hub_row(pk, bid_field, key_value, hub_load_date, record_source))
        return

    if operation["type"] == "sat":
        sat_key = ids[operation["hash_map"]].get(row.get(operation["key_field"], ""))
        if not sat_key:
            return
        sat_row = {schemas[operation["table"]][0]: sat_key, "load_date": sat_load_date}
        for out_field, in_field in operation["mappings"].items():
            sat_row[out_field] = row.get(in_field, "")
        out[operation["table"]].append(sat_row)
        return

    left_value = resolve_hash_key(ids, row, operation["left"])
    right_value = resolve_hash_key(ids, row, operation["right"])
    if not left_value or not right_value:
        return
    out[operation["table"]].append(link_row(operation["pk"], operation["left"]["column"], left_value, operation["right"]["column"], right_value, link_load_date, record_source))



In [0]:
def build_silver(input_catalog, input_schema, output_catalog, output_schema,
                 metadata_catalog, metadata_schema, metadata_table,task_run_id,full_table_name,
                 hub_load_date=None, link_load_date=None, sat_load_date=None, record_source="CRM"):
    
    runtime = load_runtime_metadata(metadata_catalog, metadata_schema, metadata_table, record_source=record_source)

    task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"Reading from {input_catalog}.{input_schema} ...")

    now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    hub_load_date = hub_load_date or now_ts
    link_load_date = link_load_date or now_ts
    sat_load_date = sat_load_date or now_ts

    if runtime["mode"] == "sql":
        spark.sql(f"USE CATALOG `{input_catalog}`")
        spark.sql(f"USE SCHEMA `{input_schema}`")

        task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"Writing to {output_catalog}.{output_schema} (merge: insert new, skip existing) ...")

        for operation in runtime["operations"]:
            table_name = operation["target_table"]
            sql_text = render_build_sql(
                operation["build_sql"],
                hub_load_date,
                link_load_date,
                sat_load_date,
                operation.get("record_source") or record_source,
            )
            result_df = spark.sql(sql_text)
            schema = result_df.columns
            key_fields = [field.strip() for field in (operation.get("merge_keys") or "").split("|") if field.strip()] or [field for field in schema if field.endswith("_hash_key")] or [schema[0]]
            rows = dedupe(collect_rows(result_df), key_fields)
            write_delta_table(output_catalog, output_schema, table_name, rows, schema, key_columns=key_fields)
            task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"  ? {table_name}: {len(rows)} rows")

        task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"
Silver vault written to {output_catalog}.{output_schema}")
        return f"{output_catalog}.{output_schema}"

    raw_tables = runtime["raw_tables"]
    schemas = runtime["schemas"]
    hub_schemas = runtime["hub_schemas"]
    hash_key_specs = runtime["hash_key_specs"]
    transform_specs = runtime["transform_specs"]

    raw = {alias: read_table_rows(input_catalog, input_schema, table_name) for alias, table_name in raw_tables.items()}
    ids = {alias: hk_map(raw[spec["source"]], spec["field"]) for alias, spec in hash_key_specs.items()}
    out = {name: [] for name in schemas}

    for spec in transform_specs:
        for row in raw.get(spec["source"], []):
            for operation in spec["operations"]:
                apply_operation(row, operation, ids, out, hub_schemas, schemas, hub_load_date, link_load_date, sat_load_date, record_source)

    task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"Writing to {output_catalog}.{output_schema} (merge: insert new, skip existing) ...")

    for table_name, schema in schemas.items():
        key_fields = [field for field in schema if field.endswith("_hash_key")] or [schema[0]]
        rows = ordered(dedupe(out[table_name], key_fields), schema)
        write_delta_table(output_catalog, output_schema, table_name, rows, schema, key_columns=key_fields)
        task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"  ? {table_name}: {len(rows)} rows")

    task_log(task_run_id=task_run_id,task_name=full_table_name,message=f"
Silver vault written to {output_catalog}.{output_schema}")
    return f"{output_catalog}.{output_schema}"




In [0]:
def get_param(widget_name, fallback):
    value = dbutils.widgets.get(widget_name)
    if value == "":
        return fallback
    return value

metadata_catalog = get_param("metadata_catalog", METADATA_CATALOG)
metadata_schema = get_param("metadata_schema", METADATA_SCHEMA)
metadata_table = get_param("metadata_table", METADATA_TABLE)
hub_load_date = get_param("hub_load_date", "")
link_load_date = get_param("link_load_date", "")
sat_load_date = get_param("sat_load_date", "")

log_message = 'Reading bronze delta tables'
task_run_id = '999'
full_table_name = f'{metadata_catalog}.{metadata_schema}.{metadata_table}'

source_configs = load_source_configs(metadata_catalog, metadata_schema, metadata_table)

required_params = {
    "metadata_catalog": metadata_catalog,
    "metadata_schema": metadata_schema,
    "metadata_table": metadata_table,
}
missing = [k for k, v in required_params.items() if not v]
if missing:
    task_log(task_run_id=task_run_id, task_name=full_table_name, message=f"Missing required parameters: {', '.join(missing)}")
    raise ValueError(f"Missing required parameters: {', '.join(missing)}")

task_log(task_run_id=task_run_id, task_name=full_table_name, message=log_message)
task_log(task_run_id=task_run_id, task_name=f'{metadata_catalog}.{metadata_schema}.{metadata_table}', message='Metadata tables..')

results = []

for source_config in source_configs:
    source_full_table_name = f"{source_config['input_catalog']}.{source_config['input_schema']}"
    source_output_name = f"{source_config['output_catalog']}.{source_config['output_schema']}"
    task_log(task_run_id=task_run_id, task_name=source_full_table_name, message=f"{log_message} [{source_config['source_name']}]")
    task_log(task_run_id=task_run_id, task_name=source_output_name, message=f"Output silver tables [{source_config['source_name']}]")

    result = build_silver(
        source_config['input_catalog'],
        source_config['input_schema'],
        source_config['output_catalog'],
        source_config['output_schema'],
        metadata_catalog,
        metadata_schema,
        metadata_table,
        task_run_id,
        source_full_table_name,
        hub_load_date,
        link_load_date,
        sat_load_date,
        source_config['record_source']
    )
    results.append((source_config['source_name'], result))
    task_log(task_run_id=task_run_id, task_name=source_full_table_name, message=f"Done → {result}")

task_log(task_run_id=task_run_id, task_name=full_table_name, message=f"Completed sources: {', '.join(name for name, _ in results)}")
